In [ ]:
import pandas as pd
import numpy as np

path1 = "댓글bert학습용.xlsx"

df = pd.read_excel(path1)
df = df[['naver_article_id','comment','label']]
df = df.dropna()

print(len(df))
df.head(2)

In [ ]:
import pandas as pd
import numpy as np

path1 = "댓글bert학습용.xlsx"

df = pd.read_excel(path1)
df = df[['naver_article_id','comment','label']]
df = df.dropna()

print(len(df))
df.head(2)

In [ ]:
df['label'].value_counts()

In [ ]:
df['label'].isnull().sum()

In [ ]:
# 1. seed 고정
# -----------------------
import random
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# 2. 모델 설정
# -----------------------
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

model_name = "beomi/KcBERT-base"
num_labels = 2
batch_size = 16
epochs = 3
max_length = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. 역비중 가중치 적용
# -----------------------
import torch

# 클래스 샘플 수
n0 = len(df[df['label'] == 0])
n1 = len(df[df['label'] == 1])

N = n0 + n1
k = 2  # 클래스 개수

# 역비율 가중치
class_weights = torch.tensor([
    N / (k * n0),  # 리스크
    N / (k * n1),  # 비리스크
], dtype=torch.float)

print(class_weights)

# GPU / CPU 이동
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Loss (가중치 적용)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# 4. 모델 대상 데이터
# -----------------------
texts = df["comment"].tolist()
labels = df["label"].tolist()

In [ ]:
# 5. 트레인, 테스트 분리 -> 데이터 프레임 만들기
# -----------------------
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

df_train = pd.DataFrame({
    "text": train_texts,
    "label": train_labels
})

df_test = pd.DataFrame({
    "text": val_texts,
    "label": val_labels
})
print(f"트레인 데이터 : {len(df_train)}개")
print(f"테스트 데이터 : {len(df_test)}개")


In [ ]:
# 4. Dataset 클래스
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(model_name)

class CustomDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = CustomDataset(train_texts, train_labels)
val_dataset = CustomDataset(val_texts, val_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 5. 모델 로드
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# 6. 학습 + 검증
# -----------------------
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
            )

        logits = outputs.logits
        # 🔥 weighted loss 적용
        loss = criterion(logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"\nEpoch {epoch+1} Train Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# 7. Validation 평가
# -----------------------
from sklearn.metrics import accuracy_score, f1_score

model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:,0].cpu().numpy())  # 0 = 리스크 확률

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation Macro F1: {f1:.4f}")

print("\n✅ 모델 평가 완료")

In [ ]:
# 7. Validation 평가 - threshold
# -----------------------
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.3, 0.71, 0.05)

results = []

for t in thresholds:

    preds = np.where(all_probs >= t, 0, 1)

    precision0 = precision_score(all_labels, preds, pos_label=0)
    recall0 = recall_score(all_labels, preds, pos_label=0)
    f10 = f1_score(all_labels, preds, pos_label=0)

    precision1 = precision_score(all_labels, preds, pos_label=1)
    recall1 = recall_score(all_labels, preds, pos_label=1)
    f11 = f1_score(all_labels, preds, pos_label=1)

    results.append([t, precision0, recall0, f10, precision1, recall1, f11])

df_threshold = pd.DataFrame(
    results,
    columns=[
        "threshold",
        "precision_0",
        "recall_0",
        "f1_0",
        "precision_1",
        "recall_1",
        "f1_1"
    ]
)

print(df_threshold)

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

# 1. 모델을 CPU로 잠시 옮겨서 안전하게 확률 추출 (MPS 에러 방지)
model.to("cpu")
model.eval()

all_probs = []
all_labels_val = []

print("최종 모델의 확률값을 추출하는 중...")
with torch.no_grad():
    for batch in val_loader:
        # 평가 데이터도 CPU로 이동
        input_ids = batch["input_ids"].to("cpu")
        attention_mask = batch["attention_mask"].to("cpu")

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Softmax를 통해 '리스크(1)'일 확률만 추출
        probs = F.softmax(outputs.logits, dim=1)
        risk_probs = probs[:, 1].numpy()

        all_probs.extend(risk_probs)
        all_labels_val.extend(batch["labels"].cpu().numpy())

# 2. 임계값 그리드 서치 (0.3 ~ 0.7)
thresholds = np.arange(0.3, 0.71, 0.05)
results = []

print("\n" + "="*60)
print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'F2-Score':<10}")
print("-" * 60)

for t in thresholds:
    # 확률값이 t 이상이면 1(리스크), 아니면 0
    preds = (np.array(all_probs) >= t).astype(int)

    # 💡 pos_label=1 (리스크) 기준 지표 계산
    p = precision_score(all_labels_val, preds, zero_division=0)
    r = recall_score(all_labels_val, preds, zero_division=0) # 우리가 보고 싶던 Recall 1
    f1 = f1_score(all_labels_val, preds, zero_division=0)
    f2 = fbeta_score(all_labels_val, preds, beta=2, zero_division=0) # 리스크 탐지용

    results.append([t, p, r, f1, f2])
    print(f"{t:<10.2f} | {p:<10.3f} | {r:<10.3f} | {f1:<10.3f} | {f2:<10.3f}")

# 3. 데이터프레임 정리 및 모델 원상복구
df_res = pd.DataFrame(results, columns=["threshold", "precision", "recall", "f1", "f2"])
model.to(device) # 다시 MPS/GPU로 복구

print("="*60)
print("\n[전략적 추천] 리콜(Recall)이 가장 높으면서 정밀도가 무너지지 않는 지점을 찾으세요.")

In [ ]:
from sklearn.metrics import confusion_matrix

threshold = 0.65 # Precision과 Recall의 균형을 고려

y_true = np.array(all_labels)      # 🔥 그대로 사용
y_score = np.array(all_probs)

y_pred = (y_score >= threshold).astype(int)  # 🔥 방향 통일

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

In [ ]:
print(min(all_probs), max(all_probs))

In [ ]:
# PR Curve
# -----------------------
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt


# 리스크 확률
risk_probs = np.array(all_probs)
risk_labels = (np.array(all_labels) == 1).astype(int)

precision, recall, _ = precision_recall_curve(risk_labels, risk_probs)
pr_auc = auc(recall, precision)

plt.figure()
plt.plot(recall, precision)
plt.xlabel("Recall (Risk)")
plt.ylabel("Precision (Risk)")
plt.title(f"PR Curve (Risk=1) AUC={pr_auc:.3f}")
plt.show()

In [ ]:
# Lift Chart
# -----------------------
df_lift = pd.DataFrame({
    "risk_prob": np.array(all_probs),
    "label": (np.array(all_labels) == 1).astype(int)
})

df_lift = df_lift.sort_values("risk_prob", ascending=False).reset_index(drop=True)

df_lift["cum_risk"] = df_lift["label"].cumsum()
df_lift["population"] = np.arange(1, len(df_lift) + 1)

baseline = df_lift["label"].mean()
df_lift["lift"] = (df_lift["cum_risk"] / df_lift["population"]) / baseline

plt.figure()
plt.plot(df_lift["population"] / len(df_lift), df_lift["lift"])
plt.xlabel("Population %")
plt.ylabel("Lift")
plt.title("Lift Chart (Risk=1)")
plt.show()

In [ ]:
# Expected Profit
# -----------------------
from sklearn.metrics import confusion_matrix

risk_probs = np.array(all_probs)

thresholds = np.arange(0.3, 0.71, 0.05)
profits = []

benefit_tp = 5
cost_fp = 1
cost_fn = 10

risk_labels = (np.array(all_labels) == 1).astype(int)

for t in thresholds:
    preds = (risk_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(risk_labels, preds).ravel()
    profit = tp * benefit_tp - fp * cost_fp - fn * cost_fn
    profits.append(profit)

plt.figure()
plt.plot(thresholds, profits)
plt.xlabel("Threshold")
plt.ylabel("Expected Profit")
plt.title("Expected Profit Curve (Risk=1)")
plt.show()

전체에 적용

In [ ]:
df_carti = pd.read_excel("bertopic3.xlsx")
print(f"까르띠에 리스크 게시글 수 {len(df_carti)} 개")

In [ ]:
df_carti.columns

explode(x) -> 리스트게시글에만

In [ ]:
import torch
import numpy as np
from tqdm import tqdm

model.eval()
model.to(device)

risk_lists = []
risk_ratios = []

for text in tqdm(df_carti["comments"]):
    comments = str(text).split("|")
    preds_list = []

    for c in comments:
        encoding = tokenizer(
            c.strip(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        probs = torch.softmax(outputs.logits, dim=1)
        risk_prob = probs[:,1].item()  # 리스크=1 확률
        pred = 1 if risk_prob >= 0.65 else 0

        preds_list.append(pred)

    risk_lists.append(preds_list)
    risk_ratios.append(sum(preds_list) / len(preds_list))

df_carti["comment_risk_list"] = risk_lists
df_carti["comment_risk_ratio"] = risk_ratios

In [ ]:
df_carti.head(5)

In [ ]:
df_carti.to_excel("리스크댓글_고도화ver최종.xlsx")
print(f"까르띠에 댓글 분석 : {len(df_carti)}개 완료")

In [ ]:
df_carti['comment_risk_ratio'].describe()

In [ ]:
len(df_carti[df_carti['comment_risk_ratio'] >= 0.5])